# Sentiment-Conditioned Trader DNA

A standout take on the PrimeTrade assignment: instead of stopping at a simple fear-vs-greed correlation, this notebook builds a **trader DNA framework** that asks four questions:

1. Which sentiment regimes produce the best realized outcomes?
2. Are the best traders trend-followers or contrarians?
3. Do regime transitions matter more than regime levels?
4. Can we segment wallets into repeatable behavioral archetypes?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

trades = pd.read_csv('/mnt/data/historical_data.csv')
sentiment = pd.read_csv('/mnt/data/Pasted text.txt', sep='\t')
trades['date'] = pd.to_datetime(trades['Timestamp IST'], format='%d-%m-%Y %H:%M').dt.normalize()
sentiment['date'] = pd.to_datetime(sentiment['date']).dt.normalize()
print(trades.shape, sentiment.shape)
print('wallets:', trades['Account'].nunique(), 'coins:', trades['Coin'].nunique())
print('date range:', trades['date'].min().date(), 'to', trades['date'].max().date())

In [ ]:
def map_exposure(direction, side):
    d = str(direction).lower()
    s = str(side).lower()
    if d in {'buy', 'open long', 'close short', 'settlement'} or 'long > short' in d:
        return 'Long'
    if d in {'sell', 'open short', 'close long', 'spot dust conversion'} or 'short > long' in d or 'auto-deleveraging' in d or 'liquidated isolated short' in d:
        return 'Short'
    if 'close short' in d or ('short' not in d and s == 'buy'):
        return 'Long'
    return 'Short'


df = trades.merge(sentiment[['date', 'value', 'classification']], on='date', how='left').dropna(subset=['classification']).copy()
df['exposure_side'] = [map_exposure(d, s) for d, s in zip(df['Direction'], df['Side'])]
df['abs_notional'] = df['Size USD'].abs()
df['is_close'] = df['Closed PnL'] != 0
df['is_win'] = df['Closed PnL'] > 0
df['roi_trade'] = df['Closed PnL'] / df['abs_notional'].replace(0, np.nan)
closed = df[df['is_close']].copy()
print('matched rows:', len(df), 'closing trades:', len(closed))

## 1) Regime leaderboard

In [ ]:
regime = closed.groupby('classification').agg(
    close_trades=('Account', 'size'),
    total_pnl=('Closed PnL', 'sum'),
    avg_pnl=('Closed PnL', 'mean'),
    win_rate=('is_win', 'mean'),
    avg_roi=('roi_trade', 'mean'),
    gross_volume=('abs_notional', 'sum'),
).sort_values('avg_pnl', ascending=False)
regime['pnl_per_million_volume'] = regime['total_pnl'] / regime['gross_volume'] * 1_000_000
regime.round(4)

In [ ]:
plot_df = regime.sort_values('avg_pnl')
fig, ax = plt.subplots(figsize=(9,5))
ax.barh(plot_df.index, plot_df['avg_pnl'])
ax.set_title('Average Realized PnL per Closing Trade by Sentiment')
ax.set_xlabel('USD')
plt.tight_layout()

## 2) Contrarian edge matrix

In [ ]:
side_regime = closed.groupby(['classification', 'exposure_side']).agg(
    close_trades=('Account', 'size'),
    total_pnl=('Closed PnL', 'sum'),
    avg_pnl=('Closed PnL', 'mean'),
    win_rate=('is_win', 'mean'),
    avg_roi=('roi_trade', 'mean'),
).reset_index()
side_regime.round(4)

In [ ]:
heat = side_regime.pivot(index='classification', columns='exposure_side', values='avg_roi').reindex(
    ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
)
fig, ax = plt.subplots(figsize=(7,5))
im = ax.imshow(heat.values, aspect='auto')
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat.iloc[i,j]:.2%}", ha='center', va='center')
ax.set_title('Average ROI by Regime and Direction')
plt.colorbar(im, ax=ax)
plt.tight_layout()

## 3) Transition analysis

In [ ]:
daily = closed.groupby('date').agg(
    total_pnl=('Closed PnL', 'sum'),
    trades=('Account', 'size'),
    avg_roi=('roi_trade', 'mean'),
).reset_index().merge(sentiment[['date', 'classification', 'value']], on='date', how='left').sort_values('date')
daily['prev_class'] = daily['classification'].shift(1)
daily['transition'] = daily['prev_class'].fillna('Start') + ' -> ' + daily['classification']
transitions = daily.groupby('transition').agg(days=('date','size'), avg_daily_pnl=('total_pnl','mean'), avg_roi=('avg_roi','mean')).reset_index()
transitions[transitions['days'] >= 2].sort_values('avg_daily_pnl', ascending=False).head(10).round(4)

## 4) Wallet archetypes

In [ ]:
acct = closed.groupby('Account').agg(
    total_pnl=('Closed PnL', 'sum'),
    trades=('Closed PnL', 'size'),
    win_rate=('is_win', 'mean'),
    avg_roi=('roi_trade', 'mean'),
    short_share=('exposure_side', lambda s: (s == 'Short').mean()),
    active_days=('date', 'nunique'),
    coins=('Coin', 'nunique'),
    total_volume=('abs_notional', 'sum'),
).reset_index()

X = np.column_stack([
    np.log1p(acct['trades']),
    np.log1p(acct['active_days']),
    np.log1p(acct['coins']),
    np.log1p(acct['total_volume']),
    acct['win_rate'],
    acct['avg_roi'].fillna(0),
    acct['short_share'],
])
X = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=4, random_state=42, n_init=50)
acct['cluster'] = km.fit_predict(X)
cluster_summary = acct.groupby('cluster').agg(
    traders=('Account', 'size'),
    avg_wallet_pnl=('total_pnl', 'mean'),
    avg_trades=('trades', 'mean'),
    avg_win_rate=('win_rate', 'mean'),
    avg_roi=('avg_roi', 'mean'),
    avg_short_share=('short_share', 'mean'),
    avg_coins=('coins', 'mean'),
    avg_volume=('total_volume', 'mean'),
).round(4)
cluster_summary

## Final takeaway

This dataset does **not** say “buy when fear, sell when greed” in a simplistic way.

It says something more interesting:

- **Extreme Greed** created the richest realized exits per trade.
- **Fear** created the largest absolute profit pool because more trades got closed there.
- **Short-side positioning consistently monetized euphoric regimes better than long-side positioning.**
- The strongest signal was often a **regime shift**, not a static sentiment level.
- The wallet population is not homogeneous — it contains at least four distinct trading archetypes.

That turns the assignment from a plain EDA into a **behavioral intelligence framework**.
